# 3. All Important Model Parameters

**Practical use case:** Model, temperature, top-p, length, penalties, retries, timeout, stop, seed, logprobs and reasoning effort.

This trainer-ready notebook contains explanation, live `langchain_openai` code, validation guidance and exercises. It makes real API calls and may incur charges.

## 1. Business problem

**Scenario:** Experiment with creativity, length, repetition, reliability and streaming settings.

The goal is to convert an unstructured language task into a repeatable workflow that can be demonstrated, tested and later integrated into an application.

## 2. Solution workflow

1. Prepare or load the input.
2. Define the model and important parameters.
3. Construct a precise prompt or schema.
4. Invoke the model.
5. inspect and validate the response.
6. Save or pass the result to the next application step.

## Parameters covered

- `model`: model identifier
- `temperature`: randomness; lower is more consistent
- `top_p`: nucleus sampling
- `max_completion_tokens`: output limit
- `stop`: stop sequences
- `frequency_penalty`: discourages repeated frequency
- `presence_penalty`: encourages new topics
- `timeout` and `max_retries`: reliability controls
- `streaming`: token-by-token delivery
- `seed`: best-effort reproducibility where supported
- `logprobs`: token probability information where supported
- `reasoning_effort`: applicable to supported reasoning models

OpenAI does not expose `top_k` as a standard ChatOpenAI generation parameter.

### 1. Set up the API key and model

This cell imports the required classes, reads the API key securely, and selects the model without exposing credentials.

**Expected result:** No model output is produced; the environment becomes ready for later API calls. Read the output before continuing to the next cell.

In [ ]:
import os, getpass
from langchain_openai import ChatOpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ")

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

### 2. Call the model and inspect the response

This cell calls the configured model with the prepared prompt and extracts the returned content.

**Expected result:** A live model response is printed; wording can vary unless deterministic settings are used. Read the output before continuing to the next cell.

In [ ]:
configs = [
    {"name":"deterministic", "temperature":0, "top_p":1.0},
    {"name":"balanced", "temperature":0.5, "top_p":0.9},
    {"name":"creative", "temperature":1.0, "top_p":0.95},
]

for cfg in configs:
    llm = ChatOpenAI(
        model=MODEL_NAME,
        temperature=cfg["temperature"], top_p=cfg["top_p"],
        max_completion_tokens=120, frequency_penalty=0.2,
        presence_penalty=0.1, timeout=30, max_retries=2,
        seed=42
    )
    result = llm.invoke("Suggest three names for an AI-based customer support application.")
    print(f"\n--- {cfg['name']} ---\n{result.content}")

### 3. Call the model and inspect the response

This cell calls the configured model with the prepared prompt and extracts the returned content.

**Expected result:** A live model response is printed; wording can vary unless deterministic settings are used. Read the output before continuing to the next cell.

In [ ]:
# Stop sequence demonstration
llm_stop = ChatOpenAI(model=MODEL_NAME, temperature=0, stop=["END"])
print(llm_stop.invoke("List three benefits of RAG, then write END.").content)

## Reasoning effort and log probabilities

`reasoning_effort` applies only to models that support reasoning controls. Some reasoning models restrict sampling parameters. `logprobs=True` provides token-level probability information when the selected model supports it.

### 4. Call the model and inspect the response

This cell calls the configured model with the prepared prompt and extracts the returned content.

**Expected result:** A live model response is printed; wording can vary unless deterministic settings are used. Read the output before continuing to the next cell.

In [ ]:
# Optional reasoning-model example
reasoning_llm = ChatOpenAI(
    model=os.getenv("OPENAI_REASONING_MODEL", "gpt-5-nano"),
    reasoning_effort="low",
    max_completion_tokens=200,
    timeout=60,
    max_retries=2
)
print(reasoning_llm.invoke("Compare classification and regression in a compact table.").content)

### 5. Call the model and inspect the response

This cell calls the configured model with the prepared prompt and extracts the returned content.

**Expected result:** A live model response is printed; wording can vary unless deterministic settings are used. Read the output before continuing to the next cell.

In [ ]:
# Optional log-probability example for a compatible model
logprob_llm = ChatOpenAI(model=MODEL_NAME, temperature=0, logprobs=True)
message = logprob_llm.invoke("Return one word: Positive, Negative, or Neutral. Text: The service was excellent.")
print(message.content)
print(message.response_metadata.get("logprobs", "Log probabilities were not returned."))